In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import json
import numpy as np
import os
import glob

from pathlib import Path, PosixPath
from typing import Dict, List, Tuple

In [ ]:
def get_image_in_folder(path_to_image: str, suffix: List[str] = [".png", ".jpeg", ".jpg", ".tif", ".tiff"]) -> List[PosixPath]:
    """
    Search for image paths and image names.

    param:
        directory: str
            Directory for image search.

    returns:
        image_paths: list[PosixPath]
            List of image paths.
    """
    return [file_name for file_name in Path(path_to_image).rglob("*")
                   if file_name.suffix in suffix and ".ipynb_checkpoints" not in file_name.as_posix()]


def get_json_coordinate(path_json: str, preffix_image: str) -> Dict[str, List[str]]:
    """
   Get image coordinates

    Args:
        image_path: path to images
        camera_type: type of camera, "bottom" or "top"

    Returns:
        points: points for mapping
    """

    image_characteristic = {
        "image_main": [],
        "image_second": [],
        "points_main": [],
        "points_second": []
    }
    
    with open(path_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for idx in range(len(data)):
        points_main = []
        points_second = []

        variable_main = data[idx]["file1_path"].split('/')
        variable_second = data[idx]["file2_path"].split('/')

        image_characteristic["image_main"].append(f"{preffix_image}/{variable_main[-3]}/{variable_main[-2]}/{variable_main[-1]}")
        image_characteristic["image_second"].append(f"{preffix_image}/{variable_second[-3]}/{variable_second[-2]}/{variable_second[-1]}")

        for p in zip(data[idx]["image1_coordinates"], data[idx]["image2_coordinates"]):
            points_main.append([p[0]["x"], p[0]["y"]])
            points_second.append([p[1]["x"], p[1]["y"]])

        image_characteristic["points_main"].append(np.array(points_main))
        image_characteristic["points_second"].append(np.array(points_second))

    return image_characteristic

# В каком виде лучше хранить данные и отправлять в findHomography?

In [ ]:
train_json = get_image_in_folder("train", suffix = [".json"])
val_json = get_image_in_folder("val", suffix = [".json"])

In [ ]:
image_characteristic = get_json_coordinate(train_json[0], "train")

In [ ]:
all_image_characteristic = {}

for jsn in train_json:
    if 'bottom' in jsn.as_posix():
        image_characteristic = get_json_coordinate(jsn, "train")
        all_image_characteristic[jsn] = image_characteristic

In [ ]:
all_src = []
all_dst = []

for v in all_image_characteristic.values():
    for idx in range(len(v['points_second'])):
        src_pts = v['points_second'][idx]
        dst_pts = v['points_main'][idx]
        
        if len(src_pts) > 0 and len(dst_pts) > 0:
            all_src.append(src_pts)
            all_dst.append(dst_pts)

src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst)        

# src_norm, dst_norm = _normalize_coordinates(src_all, dst_all)

# H, mask = cv2.findHomography(src_all, dst_all, 
#                              method=cv2.RANSAC, 
#                              ransacReprojThreshold=5.0, 
#                              maxIters=2000, 
#                              confidence=0.995)
H, mask = cv2.findHomography(src_all, dst_all, cv2.LMEDS, 12.0, 
                             maxIters=2000, 
                             confidence=0.995)
# H, mask = cv2.findHomography(src_all, dst_all, cv2.RHO, 5.0)

print("Homography Matrix:")
print(H)

In [ ]:
metric = batch_euclidean_distance(points_transformed, dst_all)

In [ ]:
l = 0
for jsn_pth in all_image_characteristic:
    all_image_characteristic[jsn_pth]['metrcis'] = metric[l: l + len(all_image_characteristic[jsn_pth]['image_second'])]
    l += len(all_image_characteristic[jsn_pth]['image_second'])

In [ ]:
best_metric = 0
main_path = ''
second_path = ''
besat_point = []
for jsn_pth in all_image_characteristic:
    for idx in range(len(all_image_characteristic[jsn_pth]['metrcis'])):
        if all_image_characteristic[jsn_pth]['metrcis'][idx] > best_metric:
            best_metric = all_image_characteristic[jsn_pth]['metrcis'][idx]
            main_path = all_image_characteristic[jsn_pth]['image_main'][idx]
            second_path = all_image_characteristic[jsn_pth]['image_second'][idx]
            besat_point = all_image_characteristic[jsn_pth]['points_second'][idx]
            
            break
    break

In [ ]:
second_path, jsn_pth.parent / "/".join(Path(second_path).parts[2:])

In [ ]:
"/".join(Path(second_path).parts[2:])

In [ ]:
srcPoints = image_characteristic['points_second'][0]
dstPoints = image_characteristic['points_main'][0]

H, mask = cv2.findHomography(srcPoints, dstPoints, cv2.RANSAC, 5.0)

print("Homography Matrix:")
print(H)

In [ ]:
train_images = get_image_in_folder("train")
images_count_to_visualization = 5

camera_type = {
                "bottom":[],
                "door2":[],
                "top":[]
}

for s in train_images:
    p = s.parts
    camera_type[p[2]].append(s)

random_numbers = random.sample(range(0, len(camera_type["bottom"])), images_count_to_visualization)

for idx in random_numbers:
    print(f"camera_type[door2][idx] = {camera_type["door2"][idx]}")
    img_bottom = cv2.imread(camera_type["bottom"][idx])
    img_door2 = cv2.imread(camera_type["door2"][idx])
    img_top = cv2.imread(camera_type["top"][idx])
    transformed_img = cv2.warpPerspective(img_bottom, H, (img_door2.shape[1], img_door2.shape[0]), cv2.INTER_LINEAR, cv2.BORDER_CONSTANT,)
    plt.figure(figsize=(20, 12))

    plt.subplot(4, 4, 1)
    plt.imshow(img_bottom)
    plt.subplot(4, 4, 2)
    plt.imshow(transformed_img)
    plt.subplot(4, 4, 3)
    plt.imshow(img_door2)
    plt.subplot(4, 4, 4)
    plt.imshow(img_top)
    
    plt.show()

# Как рассчитывать метрику?

In [ ]:
def batch_euclidean_distance(points1, points2):
    """Массив расстояний для каждой пары точек."""
    return np.sqrt(np.sum((points1 - points2)**2, axis=1))

def mean_euclidean_distance(pred_points, true_points):
    """MED - среднее евклидово расстояние."""
    return np.mean(batch_euclidean_distance(pred_points, true_points))

def median_euclidean_distance(pred_points, true_points):
    """Медианное евклидово расстояние (более устойчиво к выбросам)."""
    return np.median(batch_euclidean_distance(pred_points, true_points))

In [ ]:
points_transformed = []

for p in src_all:
    transformed_points = cv2.perspectiveTransform(p.reshape(1, 1, 2).astype(np.float32), H)
    points_transformed.append(transformed_points.squeeze())

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

In [ ]:
def draw_points_on_image(image: np.array, points_list: list, color=(0, 0, 255), radius=5, thickness=20):
    """
   Draws points on an image by coordinates

    Args:
        image: image (numpy array)
        points_list: a list of dictionaries with the keys 'x' and 'y'
        color: the point color
        radius: the radius of a point in pixels
        thickness: outline thickness

    Returns:
        image_with_points: image with drawn dots
    """
    image_points = image.copy()

    for number, point in enumerate(points_list):
        x, y = point.astype(int)
        cv2.circle(image_points, (x, y), radius, color, thickness)

        cv2.putText(image_points, str(number + 1), 
                   (x + radius, y - radius), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    return image_points


def get_json_coordinate_val(camera_type: str, image_path: str) -> list:
    """
   Get image coordinates

    Args:
        image_path: path to images
        camera_type: type of camera, "bottom" or "top"

    Returns:
        points: points for mapping
    """
    points = []
    if "bottom" == camera_type:
        path_json = image_path.parent.parent / "coords_bottom.json"
    else:
        path_json = image_path.parent.parent / "coords_top.json"
    
    if path_json.exists():
        with open(path_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        return False
    for idx in range(len(data)):

        if image_path.name in data[idx]['file1_path']:
            points = data[idx]['image1_coordinates']
            break

        if image_path.name in data[idx]['file2_path']:
            points = data[idx]['image2_coordinates']
            break

    return points

In [ ]:
dataset = "train"
val_json = get_image_in_folder(dataset, suffix = [".json"])
val_images = get_image_in_folder(dataset)
images_count_to_visualization = 30
random_numbers = random.sample(range(0, len(val_json)), images_count_to_visualization)
imctv = 0
camera_type = 'bottom'
for jsn_idx in range(len(val_json)):
    all_image_characteristic = {}
    if camera_type in val_json[jsn_idx].as_posix():
        image_characteristic = get_json_coordinate(val_json[jsn_idx], dataset)
        all_image_characteristic[val_json[jsn_idx]] = image_characteristic

        json_name = list(all_image_characteristic.keys())[0]

        image_main = all_image_characteristic[json_name]["image_main"][0]
        image_second = all_image_characteristic[json_name]["image_second"][0]
        points_second = all_image_characteristic[json_name]["points_second"][0]
        points_main = []
        for ps in points_second:

            transformed_points = cv2.perspectiveTransform(ps.reshape(1, 1, 2).astype(np.float32), H)
            points_main.append(transformed_points.squeeze())
        
        for ip in val_images:
            if ip.name in image_second and camera_type == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_second = {image_second}")
                image_path_second = ip
            if ip.name in image_main and "door2" == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_main = {image_main}")
                image_path_main = ip

        img_bottom = cv2.imread(image_path_second)
        img_door2 = cv2.imread(image_path_main)

        img_bottom = draw_points_on_image(img_bottom, points_second, color=(255, 0, 0), radius=20, thickness=20)
        img_door2_bottom = draw_points_on_image(img_door2, points_main, color=(255, 0, 0), radius=20, thickness=20)
        
        plt.figure(figsize=(30, 35))
        
        plt.subplot(2, 2, 1)
        plt.imshow(img_bottom)
        plt.subplot(2, 2, 2)
        plt.imshow(img_door2_bottom)
        plt.show()
        imctv += 1
        if imctv == images_count_to_visualization:
            break

### Из поулченных резлутатов на валидационном датасете, видно, что "нормально" переносятся только центральные точки. Точки же на нижней полке, на верхних полках, на боках холодильника - переносятся "плохо".  

### Но решить проблему можно попробовать, через методы кластеризации или избавления от дисторсии. Отделяя группы ближних точек друг от друга, и для каждого кластера получить свою матрицу гомографии.

# Полиномиальная регрессия

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

In [ ]:
src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst) 

In [ ]:
X_bottom = src_all  # форма (N, 2) - координаты на bottom
y_door = dst_all

scaler_x = StandardScaler()
scaler_y = StandardScaler()

# model_x = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=0.1))
# model_y = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=0.1))

In [ ]:
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_bottom)

scaler_y_x = StandardScaler()
y_x_scaled = scaler_y_x.fit_transform(y_door[:, 0].reshape(-1, 1)).ravel()

svr_x = SVR(kernel='rbf', C=100, gamma='scale')
svr_x.fit(X_scaled, y_x_scaled)

scaler_y_y = StandardScaler()
y_y_scaled = scaler_y_y.fit_transform(y_door[:, 1].reshape(-1, 1)).ravel()

svr_y = SVR(kernel='rbf', C=100, gamma='scale')
svr_y.fit(X_scaled, y_y_scaled)

In [ ]:
points_transformed = []

for p in src_all:

    point_new = np.array([p])

    point_scaled = scaler_X.transform(point_new)

    x_door_scaled = svr_x.predict(point_scaled)[0]

    y_door_scaled = svr_y.predict(point_scaled)[0]

    x_door_pred = scaler_y_x.inverse_transform([[x_door_scaled]])[0][0]
    y_door_pred = scaler_y_y.inverse_transform([[y_door_scaled]])[0][0]


#     x_pred = svr_x.predict(x_door_pred)
#     y_pred = svr_y.predict(y_door_pred)
    point = np.array([x_door_pred, y_door_pred])
    points_transformed.append(point)
    points_transformed = [arr.reshape(-1) for arr in points_transformed]

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst) 

X_bottom = src_all  
y_door = dst_all   

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_bottom)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_door)

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 256, 64, 32, 16, 8),
    activation='relu',
    max_iter=500,
    random_state=42,
    learning_rate='adaptive',
    alpha=0.01,
    early_stopping=True,
    validation_fraction=0.1
)

mlp.fit(X_scaled, y_scaled)

In [ ]:
points_transformed = []

for p in src_all:

    point_new = np.array([p])
    point_scaled = scaler_X.transform(point_new)
    predicted = mlp.predict(point_scaled)

#     x_door, y_door = predicted[0]

#     point = np.array([x_door, y_door])
    pred = scaler_y.inverse_transform(predicted)
    points_transformed.append(pred)
    points_transformed = [arr.reshape(-1) for arr in points_transformed]

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

# Дисторсия

In [ ]:
dataset = "train"
_json = get_image_in_folder(dataset, suffix = [".json"])
_images = get_image_in_folder(dataset)

In [ ]:
all_image_characteristic = {}

for jsn in _json:
    if 'bottom' in jsn.as_posix():
        image_characteristic = get_json_coordinate(jsn, "train")
        all_image_characteristic[jsn] = image_characteristic

In [ ]:
image = cv2.imread("train/camera_door2_2025-11-27_16-52-37/door2/frame_000117.jpg", cv2.IMREAD_GRAYSCALE)
# image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)

_, binary = cv2.threshold(image, 150, 255, cv2.THRESH_BINARY)
binary = cv2.morphologyEx(binary, cv2.MORPH_ERODE, (5, 5), iterations = 1)
binary = cv2.morphologyEx(binary, cv2.MORPH_DILATE, (5, 5), iterations = 1)
plt.subplots(1, 1)
plt.imshow(image[100:1400, 400:2500])
plt.subplots(1, 1)
plt.imshow(binary[100:1400, 400:2500])

In [ ]:
img = cv2.imread('train/camera_door2_2025-11-27_16-52-37/door2/frame_000117.jpg')
h, w = img.shape[:2]

K = np.array([[w/2, 0, w/2],
              [0, w/2, h/2],
              [0, 0, 1]], dtype=np.float32)
plt.subplots(1, 1)
plt.imshow(img)
for k1 in [0.0, -0.05, -0.1,-0.13]:
    D = np.array([[k1], [0.0], [0.0], [0.0]], dtype=np.float32)
    
    K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
        K, D, (w, h), np.eye(3), balance=0.5
    )
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(
        K, D, np.eye(3), K_new, (w, h), cv2.CV_16SC2
    )
    undistorted = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)
    
    plt.subplots(1, 1)
    plt.imshow(cv2.resize(undistorted, (800, 600)))

In [ ]:
# img = cv2.imread('train/camera_door2_2025-11-27_16-52-37/door2/frame_000117.jpg')
img = cv2.imread('train/camera_door2_2025-12-19_09-49-00/door2/frame_000084.jpg')
w, h = 3200, 1800

K = np.array([[w/2, 0, w/2],
              [0, w/2, h/2],
              [0, 0, 1]], dtype=np.float32)

D = np.array([[-0.35], [0.098], [-0.021], [0.007]])
balance = 0.4 

K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
    K, D, (w, h), np.eye(3), balance=balance
)

map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    K, D, np.eye(3), K_new, (w, h), cv2.CV_16SC2
)

undistorted = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)

# plt.imshow(cv2.resize(undistorted, (800, 600)))
plt.imshow(undistorted)

In [ ]:
w, h = 3200, 1800
K = np.array([[w/2, 0, w/2],
              [0, w/2, h/2],
              [0, 0, 1]], dtype=np.float32)

D = np.array([[-0.35], [0.098], [-0.021], [0.007]])

# img = cv2.imread('train/camera_door2_2025-11-27_16-52-37/door2/frame_000117.jpg')
img = cv2.imread('train/camera_door2_2025-12-19_09-49-00/door2/frame_000084.jpg')

K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
    K, D, (w, h), np.eye(3), balance=0.3
)

map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    K, D, np.eye(3), K_new, (w, h), cv2.CV_16SC2
)

undistorted_img = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)


door2_points_distorted = np.array([
    [1611.9, 1440.2],
    [1187.5, 1435.5],
], dtype=np.float32)

door2_points_undistorted = cv2.fisheye.undistortPoints(
    door2_points_distorted.reshape(-1, 1, 2), 
    K, 
    D,
    P=K_new  
).reshape(-1, 2)

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.scatter(door2_points_distorted[:, 0], door2_points_distorted[:, 1], 
            c='red', s=50, label='Distorted points')
plt.title('Original (Fisheye)')
plt.legend()

plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB))
plt.scatter(door2_points_undistorted[:, 0], door2_points_undistorted[:, 1], 
            c='green', s=50, label='Undistorted points')
plt.title('Undistorted')
plt.legend()

plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB))
plt.scatter(door2_points_distorted[:, 0], door2_points_distorted[:, 1], 
            c='red', s=30, alpha=0.5, label='Original (wrong)')
plt.scatter(door2_points_undistorted[:, 0], door2_points_undistorted[:, 1], 
            c='green', s=30, label='Undistorted (correct)')
plt.title('Comparison')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(new_dst_all, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(new_dst_all, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(new_dst_all, dst_all)}")

In [ ]:
def undistort_fisheye_points(points, K, D, P):
    """
    Исправляет искажения fisheye для массива точек.
    
    Args:
        points: массив точек формата (N, 2) - искажённые координаты
        K: матрица камеры (3x3)
        D: коэффициенты дисторсии (4x1)
    
    Returns:
        points_undistorted: массив точек формата (N, 2) - исправленные координаты
    """

    door2_points_undistorted = cv2.fisheye.undistortPoints(
        points.reshape(-1, 1, 2), 
        K,
        D,
        P=P
    ).reshape(-1, 2)
    return door2_points_undistorted
#     points_reshaped = points.reshape(-1, 1, 2).astype(np.float32)
    
#     points_undistorted = cv2.fisheye.undistortPoints(points_reshaped, K, D)
    
#     points_undistorted = points_undistorted.reshape(-1, 2)
    
#     points_homo = np.hstack([points_undistorted, np.ones((len(points_undistorted), 1))])

#     points_pixel_homo = points_homo @ K.T

#     points_pixel = points_pixel_homo[:, :2] / points_pixel_homo[:, 2:3]
    
#     return points_pixel.astype(np.float32)

def draw_points_on_image(image: np.array, points_list: list, color=(0, 0, 255), radius=5, thickness=20):
    """
   Draws points on an image by coordinates

    Args:
        image: image (numpy array)
        points_list: a list of dictionaries with the keys 'x' and 'y'
        color: the point color
        radius: the radius of a point in pixels
        thickness: outline thickness

    Returns:
        image_with_points: image with drawn dots
    """
    image_points = image.copy()

    for number, point in enumerate(points_list):
        x, y = point.astype(int)
        cv2.circle(image_points, (x, y), radius, color, thickness)

        cv2.putText(image_points, str(number + 1), 
                   (x + radius, y - radius), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    return image_points


def get_json_coordinate_val(camera_type: str, image_path: str) -> list:
    """
   Get image coordinates

    Args:
        image_path: path to images
        camera_type: type of camera, "bottom" or "top"

    Returns:
        points: points for mapping
    """
    points = []
    if "bottom" == camera_type:
        path_json = image_path.parent.parent / "coords_bottom.json"
    else:
        path_json = image_path.parent.parent / "coords_top.json"
    
    if path_json.exists():
        with open(path_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        return False
    for idx in range(len(data)):

        if image_path.name in data[idx]['file1_path']:
            points = data[idx]['image1_coordinates']
            break

        if image_path.name in data[idx]['file2_path']:
            points = data[idx]['image2_coordinates']
            break

    return points

In [ ]:
train_images = get_image_in_folder("train")
images_count_to_visualization = 10

camera_type = {
                "bottom":[],
                "door2":[],
                "top":[]
}

for s in train_images:
    p = s.parts
    camera_type[p[2]].append(s)

random_numbers = random.sample(range(0, len(camera_type["bottom"])), images_count_to_visualization)

for idx in random_numbers:
    print(f"camera_type[door2][idx] = {camera_type["door2"][idx]}")
    img_bottom = cv2.imread(camera_type["bottom"][idx])
    img_door2 = cv2.imread(camera_type["door2"][idx])
    img_top = cv2.imread(camera_type["top"][idx])
    undistorted = cv2.remap(img_door2, map1, map2, cv2.INTER_LINEAR)

    points_bottom = get_json_coordinate("bottom", camera_type["bottom"][idx])
    points_top = get_json_coordinate("top", camera_type["top"][idx])


    plt.figure(figsize=(30, 35))

    if points_bottom != False:
        points_bottom_array = np.array([[item['x'], item['y']] for item in points_bottom])

#         points_door2_bottom = get_json_coordinate("bottom", camera_type["door2"][idx])
#         points_door2_bottom_array = np.array([[item['x'], item['y']] for item in points_door2_bottom])

        points_door2_bottom_array = []
        for p in points_bottom_array:
            transformed_points = cv2.perspectiveTransform(p.reshape(1, 1, 2).astype(np.float32), H)
            points_door2_bottom_array.append(transformed_points.squeeze())
        points_door2_bottom_array = np.array(points_door2_bottom_array)
#         points = undistort_fisheye_points(points_door2_bottom_array, K, D, K_new)
        points = cv2.fisheye.undistortPoints(
                                            points_door2_bottom_array.reshape(-1, 1, 2), 
                                            K, 
                                            D,
                                            P=K_new  
                                        ).reshape(-1, 2)
        
        img_bottom = draw_points_on_image(img_bottom, points_bottom_array, color=(255, 0, 0), radius=20, thickness=20)
        img_door2_bottom = draw_points_on_image(undistorted, points, color=(255, 0, 0), radius=20, thickness=20)
        plt.subplot(4, 4, 1)
        plt.imshow(img_bottom)
        plt.subplot(4, 4, 2)
        plt.imshow(img_door2_bottom)

    if points_top != False:
        points_top_array = np.array([[item['x'], item['y']] for item in points_top])
            
        points_door2_top = get_json_coordinate("top", camera_type["door2"][idx])
        points_door2_top_array = np.array([[item['x'], item['y']] for item in points_door2_top])
        points = undistort_fisheye_points(points_door2_top_array, K, D, K_new)
        
        img_top = draw_points_on_image(img_top, points_top_array, color=(0, 255, 0), radius=20)
        img_door2_top = draw_points_on_image(undistorted, points, color=(0, 255, 0), radius=20)
        plt.subplot(4, 4, 3)
        plt.imshow(img_door2_top)

    plt.subplot(4, 4, 4)
    plt.imshow(img_top)
    plt.show()
    points_bottom = False
    points_top = False

In [ ]:
K, D, K_new

In [ ]:
src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst)
dst_all = undistort_fisheye_points(dst_all, K, D, K_new)

# H, mask = cv2.findHomography(src_all, dst_all,
#                              method=cv2.RANSAC,
#                              ransacReprojThreshold=125,
#                              maxIters=4000,
#                              confidence=0.995)
H, mask = cv2.findHomography(src_all, dst_all, cv2.LMEDS, 6.0,
                             maxIters=4000,
                             confidence=0.995)

In [ ]:
points_transformed = []

for p in src_all:
    transformed_points = cv2.perspectiveTransform(p.reshape(1, 1, 2).astype(np.float32), H)
    points_transformed.append(transformed_points.squeeze())

In [ ]:
print(f"Расстояние для точек: {batch_euclidean_distance(points_transformed, dst_all)}")
print(f"Среднее евклидовое расстояние для точек: {mean_euclidean_distance(points_transformed, dst_all)}")
print(f"Медианное евклидовое расстояние для точек: {median_euclidean_distance(points_transformed, dst_all)}")

# Кластеризация

In [ ]:
import sklearn
from sklearn.cluster import KMeans, BisectingKMeans

In [ ]:
train_json = get_image_in_folder("train", suffix = [".json"])

In [ ]:
all_image_characteristic = {}

for jsn in train_json:
    if 'bottom' in jsn.as_posix():
        image_characteristic = get_json_coordinate(jsn, "train")
        all_image_characteristic[jsn] = image_characteristic
        
all_src = []
all_dst = []

for v in all_image_characteristic.values():
    for idx in range(len(v['points_second'])):
        src_pts = v['points_second'][idx]
        dst_pts = v['points_main'][idx]
        
        if len(src_pts) > 0 and len(dst_pts) > 0:
            all_src.append(src_pts)
            all_dst.append(dst_pts)

src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst)       

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=0, n_init="auto").fit(src_all)
bkmeans = BisectingKMeans(n_clusters=5, random_state=42, bisecting_strategy="biggest_inertia").fit(src_all)

In [ ]:
cluster = kmeans.predict(points_second)

In [ ]:
val_json = get_image_in_folder("train", suffix = [".json"])
val_images = get_image_in_folder("train")
images_count_to_visualization = 9
random_numbers = random.sample(range(0, len(val_json)), images_count_to_visualization)
wthat_class = 0
camera_type = 'bottom'
for jsn_idx in random_numbers:
    all_image_characteristic = {}
    print(f"val_json[jsn_idx] = {val_json[jsn_idx]}")
    if camera_type in val_json[jsn_idx].as_posix():
        image_characteristic = get_json_coordinate(val_json[jsn_idx], "train")
        all_image_characteristic[val_json[jsn_idx]] = image_characteristic

        json_name = list(all_image_characteristic.keys())[0]

        image_main = all_image_characteristic[json_name]["image_main"][0]
        image_second = all_image_characteristic[json_name]["image_second"][0]
        points_second = all_image_characteristic[json_name]["points_second"][0]
        points_main = []
        cluster = kmeans.predict(points_second)
        for ps in points_second:

            transformed_points = cv2.perspectiveTransform(ps.reshape(1, 1, 2).astype(np.float32), H)
            points_main.append(transformed_points.squeeze())
        
        new_points_second = [p for idx, p in enumerate(points_second) if cluster[idx] == wthat_class]
        new_points_main = [p for idx, p in enumerate(points_main) if cluster[idx] == wthat_class]
                
        for ip in val_images:
            if ip.name in image_second and camera_type == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_second = {image_second}")
                image_path_second = ip
            if ip.name in image_main and "door2" == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_main = {image_main}")
                image_path_main = ip

        img_bottom = cv2.imread(image_path_second)
        img_door2 = cv2.imread(image_path_main)

        img_bottom = draw_points_on_image(img_bottom, new_points_second, color=(255, 0, 0), radius=20, thickness=20)
        img_door2_bottom = draw_points_on_image(img_door2, new_points_main, color=(255, 0, 0), radius=20, thickness=20)
        
        plt.figure(figsize=(30, 35))
        
        plt.subplot(2, 2, 1)
        plt.imshow(img_bottom)
        plt.subplot(2, 2, 2)
        plt.imshow(img_door2_bottom)
        plt.show()

In [ ]:
all_image_characteristic = {}

for jsn in train_json:
    if 'bottom' in jsn.as_posix():
        image_characteristic = get_json_coordinate(jsn, "train")
        all_image_characteristic[jsn] = image_characteristic

all_src = []
all_dst = []

for v in all_image_characteristic.values():
    for idx in range(len(v['points_second'])):
        src_pts = v['points_second'][idx]
        dst_pts = v['points_main'][idx]

        if len(src_pts) > 0 and len(dst_pts) > 0:
            all_src.append(src_pts)
            all_dst.append(dst_pts)

src_all = np.vstack(all_src)
dst_all = np.vstack(all_dst)
num_classes = 2
H_classes = {str(num): None for num in range(num_classes)}
kmeans = KMeans(n_clusters=num_classes, random_state=0).fit(src_all)
bkmeans = BisectingKMeans(n_clusters=num_classes, random_state=42, bisecting_strategy="biggest_inertia").fit(src_all)
cluster = kmeans.predict(src_all)
for clc in H_classes.keys():
    src = [p for idx, p in enumerate(src_all) if cluster[idx] == int(clc)]
    dst = [p for idx, p in enumerate(dst_all) if cluster[idx] == int(clc)]
    H_classes[clc], matrix = cv2.findHomography(np.array(src), np.array(dst), cv2.LMEDS, 12.0)

print("Homography Matrix:")
print(H_classes)

In [ ]:
val_json = get_image_in_folder("train", suffix = [".json"])
val_images = get_image_in_folder("train")
images_count_to_visualization = 40
imctv = 0
# random_numbers = random.sample(range(0, len(val_json)), images_count_to_visualization)
wthat_class = 0
camera_type = 'bottom'

for jsn_idx in range(len(val_json)):
    all_image_characteristic = {}
    
    if camera_type in val_json[jsn_idx].as_posix():
        print(f"val_json[jsn_idx] = {val_json[jsn_idx]}")
        image_characteristic = get_json_coordinate(val_json[jsn_idx], "train")
        all_image_characteristic[val_json[jsn_idx]] = image_characteristic

        json_name = list(all_image_characteristic.keys())[0]

        image_main = all_image_characteristic[json_name]["image_main"][0]
        image_second = all_image_characteristic[json_name]["image_second"][0]
        points_second = all_image_characteristic[json_name]["points_second"][0]
        points_main = []
#         cluster = bkmeans.predict(points_second)
        for ps in points_second:
            cluster = kmeans.predict([ps])
            transformed_points = cv2.perspectiveTransform(ps.reshape(1, 1, 2).astype(np.float32), H_classes[str(cluster[0])])
            points_main.append(transformed_points.squeeze())

        for ip in val_images:
            if ip.name in image_second and camera_type == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_second = {image_second}")
                image_path_second = ip
            if ip.name in image_main and "door2" == ip.parent.name:
                print(f"ip.name = {ip.name}")
                print(f"image_main = {image_main}")
                image_path_main = ip

        img_bottom = cv2.imread(image_path_second)
        img_door2 = cv2.imread(image_path_main)

        img_bottom = draw_points_on_image(img_bottom, points_second, color=(255, 0, 0), radius=20, thickness=20)
        img_door2_bottom = draw_points_on_image(img_door2, points_main, color=(255, 0, 0), radius=20, thickness=20)
        
        plt.figure(figsize=(30, 35))
        
        plt.subplot(2, 2, 1)
        plt.imshow(img_bottom)
        plt.subplot(2, 2, 2)
        plt.imshow(img_door2_bottom)
        plt.show()
        imctv += 1
        if imctv == images_count_to_visualization:
            break